<a href="https://colab.research.google.com/github/sanjaya/aiworks/blob/main/notebooks/Google_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-Text Prompting Techniques with Google GenAI Library

**This notebook provides a collection of code samples demonstrating various text-to-text prompting techniques using the `google.genai` library. It enables direct connection to Google Cloud for hands-on experimentation with different prompting strategies.**

#Install

In [ ]:
pip install -U google-genai

#CodeSetup

In [ ]:
import os
from google.colab import userdata
from google.genai import types
import textwrap

# Automatically uses the key from Colab Secrets
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')



In [ ]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

AI works by analyzing massive amounts of data, finding patterns, and using those patterns to make predictions or decisions. 

In short: **Data in $\rightarrow$ Pattern recognized $\rightarrow$ Smart prediction out.**


In [ ]:
interaction_1 = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain how AI works in a few words",
    generation_config={
        "thinking_level": "low"
    }
)

print(interaction_1.output_text)

AI works by **learning patterns from vast amounts of data** to make smart predictions, decisions, and solve problems.


In [ ]:
print(textwrap.fill(interaction_1.output_text, width=80))

AI works by **learning patterns from vast amounts of data** to make smart
predictions, decisions, and solve problems.


*Helper Method*

In [ ]:
def get_completion(prompt, model="gemini-3.6-flash"):
    response = client.models.generate_content(
        model= model,
        contents=types.Part.from_text(text=prompt),
        config=types.GenerateContentConfig(
            temperature=0,
            top_p=0.95,
            top_k=20,
        ),
    )
    # Access the generated text directly from the response object
    return response.text

# Excercises

In [ ]:
text = f"""
You should express what you want a model to do by \
providing instructions that are as clear and \
specific as you can possibly make them. \
This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant \
or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. \
In many cases, longer prompts provide more clarity \
and context for the model, which can lead to \
more detailed and relevant outputs.
"""

prompt = f"""
Summarize the text delimited by triple backticks \
into a single sentence.
```{text}```
"""

# prompt="Explain how AI works in a few words"

response = get_completion(prompt)

#print(response)
print(textwrap.fill(response, width=80))

Providing clear, specific, and context-rich instructions—even if it results in a
longer prompt—guides a model to produce more accurate, relevant, and detailed
responses while reducing errors.


In [ ]:
#1 Ask for a structured output: JSON, HTML

prompt = f"""
Generate a list of three made-up book titles along \
with their authors and genres.
Provide them in JSON format with the following keys:
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

```json
[
  {
    "book_id": 1,
    "title": "The Clockwork Labyrinth",
    "author": "Evelyn Vane",
    "genre": "Steampunk Fantasy"
  },
  {
    "book_id": 2,
    "title": "Echoes of a Silent Star",
    "author": "Marcus Thorne",
    "genre": "Science Fiction"
  },
  {
    "book_id": 3,
    "title": "Whispers Beneath the Birch",
    "author": "Clara Montgomery",
    "genre": "Gothic Mystery"
  }
]
```


In [ ]:
#2 Ask the model to check whether conditions are satisfied

text_1 = f"""
Making a cup of tea is easy! First, you need to get some \
water boiling. While that's happening, \
grab a cup and put a tea bag in it. Once the water is \
hot enough, just pour it over the tea bag. \
Let it sit for a bit so the tea can steep. After a \
few minutes, take out the tea bag. If you \
like, you can add some sugar or milk to taste. \
And that's it! You've got yourself a delicious \
cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Get some water boiling.
Step 2 - Grab a cup and put a tea bag in it.
Step 3 - Pour the hot water over the tea bag.
Step 4 - Let it sit for a few minutes so the tea can steep.
Step 5 - Take out the tea bag.
Step 6 - Add sugar or milk to taste, if desired.


In [ ]:
#2 Ask the model to check whether conditions are satisfied - No instruction scenario.
text_2 = f"""
The sun is shining brightly today, and the birds are \
singing. It's a beautiful day to go for a \
walk in the park. The flowers are blooming, and the \
trees are swaying gently in the breeze. People \
are out and about, enjoying the lovely weather. \
Some are having picnics, while others are playing \
games or simply relaxing on the grass. It's a \
perfect day to spend time outdoors and appreciate the \
beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.


In [ ]:
#3 "Few-shot" prompting

prompt = f"""
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest \
valley flows from a modest spring; the \
grandest symphony originates from a single note; \
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
#print(response)
print(textwrap.fill(response, width=80))

<grandparent>: The willow that sways in the fiercest wind does not break; the
strongest steel is forged in the hottest flame; the most vibrant flower pushes
through the hardest stone.


In [ ]:
## Give the model time to “think”

#1 Specify the steps required to complete a task

text = f"""
In a charming village, siblings Jack and Jill set out on \
a quest to fetch water from a hilltop \
well. As they climbed, singing joyfully, misfortune \
struck—Jack tripped on a stone and tumbled \
down the hill, with Jill following suit. \
Though slightly battered, the pair returned home to \
comforting embraces. Despite the mishap, \
their adventurous spirits remained undimmed, and they \
continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions:
1 - Summarize the following text delimited by triple \
backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following \
keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

Completion for prompt 1:
1 - While climbing a hill to fetch water, siblings Jack and Jill tumbled down after an accident, but they returned home safely with their adventurous spirit intact.

2 - Alors qu'ils grimpaient une colline pour chercher de l'eau, le frère et la sœur Jack et Jill ont dégringolé suite à un accident, mais ils sont rentrés chez eux sains et saufs avec leur esprit d'aventure intact.

3 - Jack, Jill

4 - 
```json
{
  "french_summary": "Alors qu'ils grimpaient une colline pour chercher de l'eau, le frère et la sœur Jack et Jill ont dégringolé suite à un accident, mais ils sont rentrés chez eux sains et saufs avec leur esprit d'aventure intact.",
  "num_names": 2
}
```


In [ ]:
prompt_2 = f"""
Your task is to perform the following actions:
1 - Summarize the following text delimited by
  <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the
  following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)


Completion for prompt 2:
Text: <In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well. As they climbed, singing joyfully, misfortune struck—Jack tripped on a stone and tumbled down the hill, with Jill following suit. Though slightly battered, the pair returned home to comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, and they continued exploring with delight.>

Summary: While fetching water from a hilltop well, siblings Jack and Jill fell down the hill, but they returned home safely with their adventurous spirits undimmed.

Translation: Alors qu'ils allaient chercher de l'eau au sommet d'une colline, les frère et sœur Jack et Jill sont tombés, mais ils sont rentrés chez eux sains et saufs, gardant leur esprit d'aventure intact.

Names: Jack, Jill

Output JSON: {
  "french_summary": "Alors qu'ils allaient chercher de l'eau au sommet d'une colline, les frère et sœur Jack et Jill sont tombés, mais ils sont 

In [ ]:
#2 Instruct the model to work out its own solution before rushing to a conclusion

prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""
response = get_completion(prompt)
print(response)

The student's solution is **incorrect**.

### **Error Analysis:**
In step 3, the student miscalculated the variable maintenance cost:
* The problem states that maintenance costs an additional **$10 / square foot**, which should be written as **$10x$**.
* The student incorrectly wrote **$100x$** instead.

---

### **Correct Solution:**

Let $x$ be the size of the installation in square feet.

1. **Land cost:** $100x$
2. **Solar panel cost:** $250x$
3. **Maintenance cost:** $100,000 + 10x$

**Total Cost:**
$$\text{Total Cost} = 100x + 250x + 100,000 + 10x$$
$$\text{Total Cost} = 360x + 100,000$$


In [ ]:
prompt = f"""
Your task is to determine if the student's solution \
is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total.
- Then compare your solution to the student's solution \
and evaluate if the student's solution is correct or not.
Don't decide if the student's solution is correct until
you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution \
just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help \
working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations \
as a function of the number of square feet.
```
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
"""
response = get_completion(prompt)
print(response)

Actual solution:
```
Let x be the size of the installation in square feet.

Costs:
1. Land cost: $100 * x = 100x
2. Solar panel cost: $250 * x = 250x
3. Maintenance cost for the first year: $100,000 + $10 * x = 100,000 + 10x

Total cost for the first year of operations:
Total Cost = 100x + 250x + 100,000 + 10x
Total Cost = 360x + 100,000
```

Is the student's solution the same as actual solution just calculated:
```
no
```

Student grade:
```
incorrect
```


In [ ]:
## Model Limitations: Hallucinations

# Boie is a real company, the product name is not real.

prompt = f"""
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
"""
response = get_completion(prompt)
print(response)

It appears there might be a mix-up in the product name! **Boie (Boie USA)** does not actually make a product called the **"AeroGlide UltraSlim Smart Toothbrush."** 

Boie is known specifically for **manual, non-electric, eco-friendly toothbrushes** made from medical-grade antimicrobial elastomer, rather than "smart" or electronic devices. 

However, it is easy to see why this confusion happens. Boie’s manual toothbrushes feature an **ultra-slim, futuristic, lightweight design** that looks like a high-tech gadget. 

Here is a breakdown of what **Boie actually offers**, as well as what product you might actually be thinking of if you are looking for an ultra-slim smart toothbrush:

---

### What Boie *Actually* Makes

If you are looking at a sleek, minimalist toothbrush from **Boie**, you are likely looking at the **Boie Fine Toothbrush** or **Boie Ergonomic Toothbrush**. 

Key features of genuine Boie toothbrushes include:

1. **Antimicrobial Bristles:** Instead of standard nylon bristl

In [ ]:
## Iterative Prompt Development

fact_sheet_chair = """
OVERVIEW
- Part of a beautiful family of mid-century inspired office furniture,
including filing cabinets, desks, bookcases, meeting tables, and more.
- Several options of shell color and base finishes.
- Available with plastic back and front upholstery (SWC-100)
or full upholstery (SWC-110) in 10 fabric and 6 leather options.
- Base finish options are: stainless steel, matte black,
gloss white, or chrome.
- Chair is available with or without armrests.
- Suitable for home or business settings.
- Qualified for contract use.

CONSTRUCTION
- 5-wheel plastic coated aluminum base.
- Pneumatic chair adjust for easy raise/lower action.

DIMENSIONS
- WIDTH 53 CM | 20.87”
- DEPTH 51 CM | 20.08”
- HEIGHT 80 CM | 31.50”
- SEAT HEIGHT 44 CM | 17.32”
- SEAT DEPTH 41 CM | 16.14”

OPTIONS
- Soft or hard-floor caster options.
- Two choices of seat foam densities:
 medium (1.8 lb/ft3) or high (2.8 lb/ft3)
- Armless or 8 position PU armrests

MATERIALS
SHELL BASE GLIDER
- Cast Aluminum with modified nylon PA6/PA66 coating.
- Shell thickness: 10 mm.
SEAT
- HD36 foam

COUNTRY OF ORIGIN
- Italy
"""

In [ ]:
prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)
print(response)

Here is a compelling product description tailored for a retail website:

---

# Italian-Designed Mid-Century Modern Office Chair

Elevate your workspace with timeless retro charm and modern ergonomic comfort. Crafted in Italy, this mid-century inspired office chair brings sophisticated style and high-performance functionality to both home offices and professional commercial environments. 

Whether you're looking for a minimal, sleek aesthetic or cozy, fully-upholstered luxury, this highly customizable task chair is designed to fit your unique style and seating needs.

---

### **Key Features**

* **Italian Craftsmanship & Mid-Century Design:** Beautifully designed as part of a cohesive family of mid-century furniture, making it easy to match with your desks, filing cabinets, and meeting tables.
* **Fully Customizable Upholstery:** Choose between a sleek plastic back with an upholstered seat (SWC-100) or full upholstery (SWC-110). Available in **10 premium fabric choices** and **6 rich 

In [ ]:
# Issue 1: The text is too long --> Limit the number of words/sentences/characters

prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)
#print(response)
print(textwrap.fill(response, width=80))


Add mid-century style to your home or business with this Italian-made office
chair. Qualified for contract use, it features a 5-wheel pneumatic adjustable
base available in stainless steel, matte black, gloss white, or chrome.
Customize with optional armrests, fabric or leather upholstery, and soft or
hard-floor casters.


In [ ]:
len(response.split())

47

In [ ]:
# Issue 2. Text focuses on the wrong details --> Ask it to focus on the aspects that are relevant to the intended audience

prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

The description is intended for furniture retailers,
so should be technical in nature and focus on the
materials the product is constructed from.

Use at most 50 words.

Technical specifications: ```{fact_sheet_chair}```
"""
response = get_completion(prompt)
#print(response)
print(textwrap.fill(response, width=80))

Italian-made and qualified for contract use, this mid-century office chair
features a 10mm cast aluminum shell with a modified nylon PA6/PA66 coating.
Constructed with a 5-wheel plastic-coated aluminum base, HD36 foam (1.8 or 2.8
lb/ft³ density), pneumatic adjustment, optional 8-position PU armrests, and
fabric or leather upholstery.


In [ ]:
# Issue 3. Description needs a table of dimensions --> Ask it to extract information and organize it in a table

prompt = f"""
Your task is to help a marketing team create a
description for a retail website of a product based
on a technical fact sheet.

Write a product description based on the information
provided in the technical specifications delimited by
triple backticks.

The description is intended for furniture retailers,
so should be technical in nature and focus on the
materials the product is constructed from.

At the end of the description, include every 7-character
Product ID in the technical specification.

After the description, include a table that gives the
product's dimensions. The table should have two columns.
In the first column include the name of the dimension.
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website.
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_chair}```
"""

response = get_completion(prompt)
print(response)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 59.870223525s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

#Image generation

In [ ]:
from IPython.display import display, HTML

display(HTML(response))

In [ ]:
from PIL import Image
import base64

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.1-flash-image",
    input="A cyberpunk futuristic city with neon signs, high quality",
)

with open("generated_image.png", "wb") as f:

    f.write(base64.b64decode(interaction.output_image.data))

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\nPlease retry in 33.154967262s.', 'code': 'too_many_requests'}}